# 12 — Auditoría del Modelo ACTUAL de PREDIA

**CRÍTICO.** Validar el 97.89% reportado.

## Introducción

**Objetivo.** Auditar el modelo en producción: algoritmo, features, metodología y validez de sus métricas.

**Fundamento teórico.** Una auditoría de modelo busca overfitting, data leakage y errores metodológicos que inflen métricas.

**Ventajas.** Determina si las métricas reportadas son confiables y reproducibles.

**Limitaciones.** Sin el código/datos de entrenamiento original, parte de la auditoría es forense/inferencial.

**Casos de uso.** Gobernanza de modelos en salud (model risk management).


In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../src'))
import warnings; warnings.simplefilter('ignore')
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
from predia_ml import config, data, evaluate, plots
pd.set_option('display.max_columns', 60)


In [2]:
aud = json.load(open(config.METRICS_DIR/'current_model_audit.json'))
print('PKL:', aud['pkl_inspection'])
print('\nAccuracy reportada (seed/metadata):', aud['reported_accuracy_seed_metadata'])
print('Nota:', aud['note_conflicting_metric_in_code'])

PKL: {'loaded': True, 'estimator_class': 'LogisticRegression', 'n_features_in': 11, 'classes': [0, 1], 'coef_shape': [1, 11], 'intercept': [7.577719514661116], 'feature_names_in': None}

Accuracy reportada (seed/metadata): 0.9789473684210527
Nota: ml-predict.ts comenta 98.42% / precision 100% / AUC 99.75% (conflicto)


### 1) Mismatch dataset ↔ modelo
El modelo actual usa `Urea`, `Cr`, `VLDL`, **ausentes** en `diabetes_dataset.csv`, y escalas de lípidos incompatibles (mmol/L vs mg/dL). El modelo en producción fue entrenado con OTRO dataset (947 muestras).

In [3]:
print('Features modelo actual:', aud['feature_mismatch']['present_in_dataset'])
print('AUSENTES en el dataset provisto:', aud['feature_mismatch']['absent_in_dataset'])

Features modelo actual: {'Gender': 'gender', 'AGE': 'age', 'HbA1c': 'hba1c', 'Chol': 'cholesterol_total', 'TG': 'triglycerides', 'HDL': 'hdl_cholesterol', 'LDL': 'ldl_cholesterol', 'BMI': 'bmi'}
AUSENTES en el dataset provisto: ['Urea', 'Cr', 'VLDL']


### 2) Demostración de fuga (CV5, Regresión Logística)
El ~98% solo es reproducible con variables **diagnósticas/fugadas**. Con features honestas de cribado, un modelo lineal queda **cerca del azar** (base rate 60%).

In [4]:
lk = aud['leakage_demonstration_cv5']
pd.DataFrame(lk).T.rename(columns={'accuracy':'Accuracy','roc_auc':'ROC AUC'})

,Accuracy,ROC AUC
hba1c_only,0.8520,0.9325
glucose_fasting_only,0.7307,0.8082
diagnostic_labs,0.8567,0.9337
leaky_stage_plus_score,0.9979,0.9974
screening_numeric_safe,0.6163,0.6121


### Conclusión de la auditoría
- El modelo en producción es una **Regresión Logística (11 features)** servida con **coeficientes hardcodeados en TypeScript**; los `.pkl` ni se cargan.
- La accuracy 97.89% (hardcodeada en el seed) **no es reproducible** sobre el dataset provisto y proviene de otro conjunto.
- `HbA1c` (criterio diagnóstico ADA) como predictor es **fuga**: por sí sola da acc≈0.85 / AUC≈0.93.
- `diabetes_stage`+`risk_score` ⇒ acc≈0.998 (fuga total).
- **Veredicto: las métricas actuales NO son confiables.**